# EC. Primer Parcial — Proyecto Práctico HPC

**Asignatura:** Cómputo de Alto Rendimiento

**Integrantes:**
1. Diego Cortes Zermeño (@DiegoZerm)
2. Cesar Ivan Ramirez Castañon (@tprogcesarramirez-dotcom)
3. Irving Nahir Ferrusca Jaimez (@irvingjaimez)

**Objetivo:** comparar la ejecución secuencial y paralela de un programa en Python que procesa una cantidad considerable de datos, midiendo el tiempo de ejecución, el *Speedup* y la eficiencia con 1, 2 y 4 workers.

## 1. Importación de Librerías y Módulos

Importamos las librerías necesarias para el procesamiento matemático, el paralelismo y la graficación de resultados.

In [1]:
import time
import math
import multiprocessing as mp
import numpy as np
import matplotlib.pyplot as plt

print("Librerías cargadas correctamente.")
print(f"Núcleos lógicos detectados en el sistema: {mp.cpu_count()}")

Librerías cargadas correctamente.
Núcleos lógicos detectados en el sistema: 16


## 2. Definición de Algoritmos (Secuencial y Paralelo)

Cada elemento $x$ del vector de datos se procesa con la función propuesta en la práctica:

$$f(x) = \sqrt{x} + x^2 + \sin(x) + \cos(x) + \log(x)$$

La función `process_chunk` recorre un bloque de datos y acumula $f(x)$ para cada elemento. Está definida en el módulo `hpc_utils.py` (y no dentro del notebook) para que los procesos del `Pool` puedan importarla también en Windows y macOS, donde `multiprocessing` crea los procesos con el método *spawn*.

La versión **secuencial** procesa todo el vector en un solo proceso y mide su tiempo con `time.perf_counter()`.

In [2]:
from hpc_utils import process_chunk


def generate_data(n_elements=25_000_000):
    """Genera un vector de datos continuos (1, 2, ..., n) para procesar."""
    return np.arange(1, n_elements + 1, dtype=np.float64)


def run_sequential(data):
    """Ejecuta el algoritmo en un solo proceso (secuencial)."""
    start = time.perf_counter()
    result = process_chunk(data)
    end = time.perf_counter()
    return end - start, result

### Paralelización con `multiprocessing.Pool`

Python tiene el *Global Interpreter Lock* (GIL), que impide que varios hilos ejecuten código Python al mismo tiempo. Por eso usamos **procesos** en lugar de hilos: cada proceso tiene su propio intérprete y puede usar un núcleo distinto del CPU.

La versión **paralela**:
1. Divide el vector en `num_workers` bloques del mismo tamaño con `np.array_split`.
2. Crea un `Pool` con `num_workers` procesos.
3. Cada proceso ejecuta `process_chunk` sobre su bloque (`pool.map`).
4. Suma los resultados parciales.

El tiempo medido incluye la creación de los procesos y el envío de los datos, que es el costo real (*overhead*) de paralelizar.

In [3]:
def run_parallel(data, num_workers):
    """Ejecuta el algoritmo dividiendo el vector entre múltiples workers (paralelo)."""
    chunks = np.array_split(data, num_workers)
    start = time.perf_counter()
    with mp.Pool(processes=num_workers) as pool:
        results = pool.map(process_chunk, chunks)
    end = time.perf_counter()
    return end - start, sum(results)

## 3. Experimentación y Benchmarking

En esta sección se ejecutan las pruebas de rendimiento midiendo el tiempo de ejecución de:
1. La versión **secuencial**.
2. La versión **paralela** con 1, 2 y 4 workers.

Cada configuración se evalúa **3 veces** para obtener los tiempos promedio, el **Speedup** ($S_p = T_1 / T_p$) y la **Eficiencia** ($E_p = S_p / p$).

In [4]:
DATA_SIZE = 25_000_000
WORKERS = [1, 2, 4]
REPETITIONS = 3

print(f"Generando vector de datos con {DATA_SIZE:,} elementos...")
dataset = generate_data(DATA_SIZE)

# Medición de la versión secuencial pura
print("\nEvaluando versión Secuencial:")
seq_times = []
for r in range(1, REPETITIONS + 1):
    t_seq, _ = run_sequential(dataset)
    seq_times.append(t_seq)
    print(f"  Prueba {r}: {t_seq:.4f} s")

# Medición paralela con Pool (1, 2 y 4 workers)
raw_results = {}
for w in WORKERS:
    print(f"\nEvaluando con Pool ({w} Worker/s):")
    times_w = []
    for r in range(1, REPETITIONS + 1):
        t_par, _ = run_parallel(dataset, num_workers=w)
        times_w.append(t_par)
        print(f"  Prueba {r}: {t_par:.4f} s")
    raw_results[w] = times_w

# Promedios y métricas HPC
t_seq_avg = np.mean(seq_times)
avg_times = [np.mean(raw_results[w]) for w in WORKERS]
T1 = avg_times[0]  # Tiempo base con Pool de 1 worker

speedups = [T1 / Tp for Tp in avg_times]
efficiencies = [Sp / p for Sp, p in zip(speedups, WORKERS)]

print(f"\nTiempo promedio secuencial puro: {t_seq_avg:.4f} s")

Generando vector de datos con 25,000,000 elementos...

Evaluando versión Secuencial:


  Prueba 1: 10.9243 s


  Prueba 2: 10.8759 s


  Prueba 3: 10.8811 s

Evaluando con Pool (1 Worker/s):


  Prueba 1: 12.0673 s


...

  Prueba 2: 6.3903 s


  Prueba 3: 6.3231 s

Evaluando con Pool (4 Worker/s):


  Prueba 1: 3.6270 s


  Prueba 2: 3.5649 s


  Prueba 3: 3.5509 s

Tiempo promedio secuencial puro: 10.8938 s


## 4. Tabla Resumen de Métricas HPC

Resumen de los tiempos promedio de ejecución, Speedup y Eficiencia calculados a partir de los datos experimentales.

In [5]:
print("=" * 65)
print(f"{'Configuración':<18}{'Tiempo Prom (s)':<18}{'Speedup (Sp)':<15}{'Eficiencia (Ep)':<14}")
print("=" * 65)
print(f"{'Secuencial':<18}{t_seq_avg:<18.4f}{'-':<15}{'-':<14}")
for w, t, s, e in zip(WORKERS, avg_times, speedups, efficiencies):
    label = f"{w} Worker(s)"
    print(f"{label:<18}{t:<18.4f}{s:<15.4f}{e*100:<13.2f}%")
print("=" * 65)

Configuración     Tiempo Prom (s)   Speedup (Sp)   Eficiencia (Ep)
Secuencial        10.8938           -              -             
1 Worker(s)       11.9728           1.0000         100.00       %
2 Worker(s)       6.3812            1.8763         93.81        %
4 Worker(s)       3.5809            3.3435         83.59        %


## 5. Visualización de Rendimiento

Generación automatizada de las gráficas de evaluación del sistema:
1. **Workers vs. Tiempo de Ejecución**
2. **Workers vs. Speedup** (medido vs. ideal)
3. **Workers vs. Eficiencia (%)**

In [6]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Workers vs Tiempo
axes[0].plot(WORKERS, avg_times, marker='o', color='crimson', linewidth=2, label='Paralelo (Pool)')
axes[0].axhline(y=t_seq_avg, color='black', linestyle='--', label='Secuencial puro')
axes[0].set_title('Workers vs. Tiempo de Ejecución')
axes[0].set_xlabel('Número de Workers')
axes[0].set_ylabel('Tiempo Promedio (s)')
axes[0].legend()
axes[0].grid(True)

# 2. Workers vs Speedup
axes[1].plot(WORKERS, speedups, marker='s', color='navy', linewidth=2, label='Medido (Sp)')
axes[1].plot(WORKERS, WORKERS, '--', color='gray', label='Ideal (Lineal)')
axes[1].set_title('Workers vs. Speedup')
axes[1].set_xlabel('Número de Workers')
axes[1].set_ylabel('Speedup')
axes[1].legend()
axes[1].grid(True)

# 3. Workers vs Eficiencia
axes[2].plot(WORKERS, [e * 100 for e in efficiencies], marker='^', color='forestgreen', linewidth=2)
axes[2].set_title('Workers vs. Eficiencia (%)')
axes[2].set_xlabel('Número de Workers')
axes[2].set_ylabel('Eficiencia (%)')
axes[2].set_ylim(0, 110)
axes[2].grid(True)

plt.tight_layout()
plt.savefig('hpc_performance_results.png', dpi=300)
plt.show()
print("Gráfica guardada exitosamente como 'hpc_performance_results.png'.")

Gráfica guardada exitosamente como 'hpc_performance_results.png'.


/tmp/ipykernel_125128/3659862732.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Análisis de Resultados

> **Equipo de la corrida:** AMD Ryzen 7 7435HS (1 socket × 8 núcleos × 2 hilos = 16 hilos lógicos), 15 GiB de RAM, Linux Mint 22.2 (kernel 7.0.0-38-generic). Todos los tiempos corresponden a esta ejecución.

### ¿La ejecución paralela fue más rápida que la secuencial?

**Sí con 2 y 4 workers; no con 1 worker.**

| Configuración | Tiempo promedio (s) | vs. secuencial (10.8938 s) |
|---|---|---|
| Secuencial | 10.8938 | — |
| 1 worker | 11.9728 | **+9.9 % más lenta** |
| 2 workers | 6.3812 | −41.4 % |
| 4 workers | 3.5809 | −67.1 % |

Con 2 y 4 workers el programa termina claramente antes. Con 1 worker fue *más lenta* que la versión secuencial: el `Pool` debe crear el proceso, serializar y enviar el chunk de datos y recolectar el resultado, y todo ese *overhead* no se recupera cuando solo hay un proceso trabajando.

### ¿Qué número de workers obtuvo el menor tiempo?

**4 workers: 3.5809 s** (pruebas: 3.6270, 3.5649 y 3.5509 s). Es 3.04× más rápido que la secuencial y 2.90× más rápido que 1 worker. La configuración es la mejor del experimento, con un Speedup de **3.3435** y una Eficiencia de **83.59 %**.

### ¿Duplicar el número de workers duplicó el rendimiento? ¿Por qué?

**No.** El escalamiento fue sublineal:

- De 1 → 2 workers: Speedup **1.8763** (no 2.0).
- De 1 → 4 workers: Speedup **3.3435** (no 4.0).
- La eficiencia cayó de 100 % → 93.81 % → 83.59 %.

Por la **Ley de Amdahl**, $S(p) = \dfrac{1}{s + (1-s)/p}$, con $p = 4$ y $S = 3.3435$ se obtiene una fracción serial estimada de $s \approx 6.5\%$; con $p = 2$ el ajuste da $s \approx 6.6\%$ (consistente). Esa porción no paralelizable —generar el vector, dividirlo en chunks, lanzar el `Pool` y sumar los parciales— limita el rendimiento máximo.

Al **overhead de `multiprocessing`** se suma que cada proceso recibe su bloque copiado/serializado a través del *pipe* (pickle), que el sistema operativo debe planificar 4 procesos sobre los mismos núcleos y que los resultados vuelven por IPC. Por eso duplicar procesos nunca duplica el rendimiento.

### ¿Por qué el problema seleccionado puede paralelizarse?

Es un caso de ***embarrassingly parallel*** (paralelismo ingenuo): cada elemento $f(x) = \sqrt{x} + x^2 + \sin(x) + \cos(x) + \ln(x)$ se calcula de forma **independiente**, sin leer ni escribir datos compartidos, sin dependencias entre iteraciones y sin necesidad de sincronización ni comunicación entre workers durante el cálculo. Solo hay una reducción al final (`sum(results)`). Por eso basta dividir el vector con `np.array_split` y aplicar `pool.map` sin riesgo de condiciones de carrera.

### ¿En qué momento agregar más workers deja de ser beneficioso?

Cuando el **marginal de overhead supera el marginal de cómputo**. Señales observadas:

1. **La eficiencia ya decae desde 2 workers** (100 % → 93.81 % → 83.59 %): cada worker adicional aporta menos tiempo ahorrado.
2. **Al superar los 8 núcleos físicos** (16 hilos lógicos) los procesos empiezan a competir por tiempo de CPU y a compartir hiperhilos, con contención de caché L3 (16 MiB compartidos).
3. **Cuando el chunk de trabajo se hace demasiado pequeño** respecto al costo de crear el proceso y copiar los datos (granularidad insuficiente), el speedup se estanca o invierte.

En este experimento, con 8 workers la curva ya estaría claramente en zona de rendimientos decrecientes; con 4 aún conviene porque el cómputo (~10.9 s secuenciales) domina sobre el overhead (~0.5 s).

### ¿Qué limitaciones tiene el hardware utilizado?

- **CPU:** AMD Ryzen 7 7435HS — 8 núcleos físicos / 16 hilos lógicos. Solo se usaron **4 workers = la mitad de los núcleos físicos**, por lo que el hardware no se explota al completo. Las frecuencias de turbo (hasta 4.55 GHz) no se sostienen en cargas multihilo prolongadas.
- **RAM:** 15 GiB totales, ~4 GiB en uso y ~11 GiB disponibles. El vector de 25 000 000 `float64` ocupa 200 MB y, al repartirse entre procesos, `multiprocessing` duplica esa memoria al copiar los chunks, aumentando la presión sobre el controlador de memoria.
- **Thermal / frecuencia:** temperatura del sensor ACPI de ~59–62 °C durante la corrida en un **portátil**, sujeto a limitaciones de potencia (PBO/STT) y a *thermal throttling* si la carga se sostiene; el *governor* del CPU observado osciló entre `powersave` y `performance`, lo que cambia los MHz disponibles entre corridas y explica parte de la varianza entre pruebas.
- **Procesos de fondo:** el entorno de escritorio Linux Mint y otros servicios compiten por núcleos, añadiendo ruido a las mediciones (visible en la variación de ±0.1 s entre repeticiones).

### ¿Este experimento representa HPC o solamente demuestra principios utilizados en HPC? Justifiquen.

**Demuestra principios de HPC; no es un sistema HPC.** Formalmente:

- **Sí aplica principios de HPC:** descomposición de datos en bloques, ejecución concurrente en múltiples procesos, medición de **Speedup** $S_p = T_1/T_p$ y **Eficiencia** $E_p = S_p/p$, identificación de la fracción serial y del *overhead*, y evaluación del escalamiento —las métricas clásicas de la disciplina.
- **No es HPC real:** se ejecuta en **una sola computadora de propósito general** (shared-memory, un nodo), sin clúster multinodo, sin memoria distribuida, sin interconexión de baja latencia (InfiniBand/RoCE), sin MPI ni escalado a miles de núcleos, sin gestor de colas/particiones, sin I/O paralelo distribuido y con un dataset (200 MB) lejos de las escalas típicas de HPC (GB–TB).

La frontera es la escala y el alcance: HPC exige clústeres, paralelismo a gran escala y software de gestión de recursos; este ejercicio es un **laboratorio controlado de un nodo** que ilustra exactamente las métricas y el razonamiento que sí se usan en HPC.
